# 팀 공통 페어트레이딩: 여기서 시작하기

이 노트북은 공통 모델을 불러옵니다. **개인 실험은 아래 설정만 바꾸세요.**

기본 실행은 가상 데이터입니다. 실제 S&P500 투자 성과를 나타내지 않습니다.
실제 데이터로 바꾸려면 다음 설정 셀의 `CONFIG_FILE`과 `DATA_DIR`을 바꾸세요.

처음 설치할 때 저장소 폴더에서 `python -m pip install -e ".[notebook]"`을 실행하고,
설치한 Python 환경을 노트북 커널로 선택합니다. 전체 거래 규칙은 `docs/specification.md`에 있습니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "run_backtest.py").exists() and (p / "src/sp500_pairs").is_dir()), None)
if ROOT is None:
    raise RuntimeError("압축을 모두 푼 저장소 안에서 이 노트북을 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display, Image
from sp500_pairs import load_config, run_backtest
from sp500_pairs.runner import run_grid
print("공통 코드 불러오기 완료")

## 1. 내 실험 설정

- `configs/demo.toml`: 가상 데이터, 2016~2018년 6개 거래 구간.
- `configs/baseline.toml`: 실제 데이터, 2016~2024년 18개 거래 구간.
- `configs/bidirectional.toml`: 실제 데이터, 회귀 양방향 비교 옵션.

`DATA_DIR = None`이면 저장소의 `data/` 폴더를 사용합니다. 개인 경로를 적었다면 공유 전에 지우세요.
실제 데이터 첫 확인은 `run.max_windows`를 `1`로 설정하면 한 구간만 실행합니다. `0`은 전체 구간입니다.
`stop_enabled = False`로 손절 규칙을 끌 수 있습니다.

In [ ]:
CONFIG_FILE = "configs/demo.toml"
DATA_DIR = None  # 실제 데이터가 있는 폴더. 예: "/path/to/data"

PARAMETERS = {
    "selection.max_per_sector": 0,  # 섹터당 페어 수 제한 없음
    "selection.max_per_stock": 0,  # 종목당 페어 수 제한 없음
    "schedule.formation_months": 12,
    "schedule.trading_months": 6,
    "strategy.entry_z": 2.0,
    "strategy.exit_z": 0.0,
    "strategy.stop_enabled": True,
    "strategy.stop_z": 3.5,
    "strategy.max_holding_days": 60,
    "run.max_windows": 0,
}

cfg = load_config(ROOT / CONFIG_FILE).with_updates(PARAMETERS)
display(pd.DataFrame([{"설정": k, "값": str(v)} for k, v in PARAMETERS.items()]))
print("데이터:", cfg["data"]["mode"], "| 평가 구간:", cfg["evaluation"]["active_split"])

## 2. 실행

팀원 모두 같은 매매·손익 계산 함수를 사용합니다. 설정, 신호일·체결일, 비용, 후보 탈락 이유가 결과 폴더에 저장됩니다.
거래 구간 시작 자본을 최대 페어 수로 나누고, 선정되지 않은 슬롯은 현금으로 남깁니다.

In [ ]:
result = run_backtest(cfg, data_dir=DATA_DIR)
print("결과 폴더:", result["output_directory"])
print("평가 표시:", result["manifest"]["evaluation_label"])
if result["manifest"]["data_mode"] == "synthetic":
    print("주의: 아래 결과는 가상 데이터의 동작 확인용입니다.")
display(pd.Series(result["summary"], name="result").to_frame())

In [ ]:
display(Image(filename=str(result["output_directory"] / "equity_and_drawdown.png")))
display(result["windows"][["window", "selected_pairs", "slot_initial_capital", "capital_end"]])
display(result["trades"].head(10))

## 3. 여러 파라미터 조합 비교 — 선택 실행

아래 값을 `True`로 바꾸면 진입·손절 기준 4개 조합을 비교합니다. 기본 예시는 가상 데이터입니다.
조합은 `configs/grid_demo.toml`에서 바꿉니다. 실제 데이터 실험은 해당 파일의 `base`를 실제 설정으로 바꿉니다.

학습·검증·시험 날짜는 아직 미정입니다. 기본 `all` 결과는 탐색용으로 표시하고, 최종 `test`에서는 grid 탐색을 차단합니다.
성공한 파라미터를 고른 뒤 같은 기간을 최종 성과로 보고하지 않습니다.

In [ ]:
RUN_GRID = False
if RUN_GRID:
    comparison, grid_folder = run_grid(ROOT / "configs/grid_demo.toml", data_dir=DATA_DIR)
    display(comparison)
    print("비교 결과:", grid_folder)
else:
    print("단일 실행 완료. 조합 비교는 RUN_GRID=True로 선택 실행합니다.")

## 팀에 공유하기

공통 설정은 그대로 두고 개인 설정 파일과 변경 이유를 공유하세요. 결과의 `effective_config.json`을 함께 보관하면 실제 적용값을 확인할 수 있습니다.

공적분 붕괴 탐지 효과를 비교할 때는 기본 파라미터와 평가 기간을 맞춥니다. 탐지 전략 연결은 `docs/detector_extension.md`를 참고하세요.
현재 기본 대응은 다음 거래일 종가 청산과 남은 구간 신규 진입 중단입니다. 계수 재추정과 PCI 전환은 후속 연구 구현 대상입니다.

## A–D 144개 실험 — 선택 실행

추정 12/24개월 × 청산 0/+0.5 × 손절 3–7σ(0.5 간격), A → B → C → D 순서입니다. 선정용 샤프는 직전 12개월 순차 모의거래, 실제 진입 3건 이상·비용 차감 샤프 > 0을 사용합니다.

`DATA_DIR`을 실제 자료 폴더로 지정하고 아래를 `True`로 바꾸세요. 144개 실험에서는 기존 위쪽 `PARAMETERS` 대신 `selection_study.toml`을 사용합니다. 아래 `study_cfg.with_updates(...)`로 공통 기간·비용 등을 바꿀 수 있습니다.

**커널을 다시 시작**해서 새 코드를 불러오세요. [전용 실험 노트북](Run_144_Experiments.ipynb)과 [상세 규칙](../docs/selection_study.md)도 함께 확인하세요. 청산 부호는 진입 방향에 맞춰 반전되고, 진입 범위는 기존의 `2 ≤ |Z| < 손절선`입니다.

합병 발표 정책을 공통 적용하며 기본 실험 기간은 2020–2025년입니다. 발표를 인지하면 신규 진입을 막고 보유 페어 양쪽을 다음 거래 가능한 종가에 청산합니다. 이전 216개 결과와 다른 폴더를 사용합니다.


In [ ]:
RUN_144_STUDY = False
if RUN_144_STUDY:
    from sp500_pairs import run_selection_study
    study_cfg = load_config(ROOT / "configs/selection_study.toml")
    # 예: study_cfg = study_cfg.with_updates({"schedule.start": "2020-01-01"})
    study_comparison, study_folder = run_selection_study(
        study_cfg, data_dir=DATA_DIR,
        output_dir=ROOT / "results/selection_study_144_no_concentration_caps",
        models=["A", "B", "C", "D"], resume=True, retry_failed=False,
    )
    display(study_comparison)
    print("비교표:", study_folder / "comparison.csv")
else:
    print("RUN_144_STUDY=True로 바꾸면 144개 실험을 순서대로 실행합니다.")
